[파일럿] ESG 91개 항목 전체(E/S/G+공시+반도체특화) 채점 - 관리자 공용 Chroma 인덱스 사용
- 대상: SK하이닉스(000660), 91개 항목 전체를 한 번에 채점 (도메인 분리 없음)
- v6이 만든 공용 Chroma DB를 읽기 전용으로 로컬에 복사해서 사용합니다.
  (v6, chroma_db 폴더는 절대 직접 수정/실행하지 않음 - 팀 공지 준수)
- esg_query_interface.ipynb의 함수(FilteredHybridRetriever, get_evidence,
  build_scoring_prompt 등)를 원문 그대로 재사용합니다.
  → build_scoring_prompt()의 내용 자체는 절대 수정하지 않고, 그 뒤에
    "채점유형별 JSON 출력 형식 지시문"만 덧붙입니다 (팀 프롬프트 일관성 유지).
- 채점 모델: gemini-3.1-flash-lite, temperature=0

[사전 준비]
- TARGET_COMPANIES에 채점할 기업/연도를 채워 넣을 것 (현재: SK하이닉스 파일럿)

코랩 노트북에 셀 단위로 나눠서 붙여넣으시면 됩니다.

In [ ]:
# ============================================================
# ===== 0. 환경설정 =====
# ============================================================
import subprocess
import sys

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "langchain", "langchain-google-genai", "langchain-chroma", "langchain-community",
    "chromadb", "kiwipiepy", "rank_bm25", "pandas", "openpyxl", "google-genai",
])

import os
import re
import math
import json
import shutil
from getpass import getpass

import pandas as pd
from kiwipiepy import Kiwi
from langchain_chroma import Chroma
from langchain_community.retrievers import BM25Retriever
from langchain_core.documents import Document
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from google import genai
from google.genai import types as genai_types
import openpyxl

from google.colab import drive
drive.mount('/content/drive')

os.environ["GOOGLE_API_KEY"] = getpass("Google AI Studio API Key 입력 (임베딩 조회 + 채점 공용): ")

DRIVE_ROOT = "/content/drive/MyDrive/ESG 팀프로젝트/2. ESG 채점 세트/인덱스 결과"

CONFIG = {
    "embedding_model": "gemini-embedding-001",
    "persist_dir": "/content/chroma_db_local",  # 로컬 작업 전용 (Drive 직접 접근 금지)
    "drive_backup_dir": "/content/drive/MyDrive/chroma_db_local",  # 팀원이 pdfplumber+LlamaParse로 새로 구축한 DB
    "extraction_diag_csv": f"{DRIVE_ROOT}/extraction_diagnostics.csv",
    "candidate_k": 30,
    "retrieval_k": 10,
    "hybrid_weights": [0.5, 0.5],
    # 채점기준표 경로 (전체 파일이든 분리본이든 load_rubric()이 자동 인식)
    "rubric_xlsx": "/content/drive/MyDrive/ESG 팀프로젝트/2. ESG 채점 세트/채점기준표/ESG_핵심지표_채점기준표_전체.xlsx",
}

JUDGE_MODEL = "gemini-3.1-flash-lite"  # GA 정식버전 (2026.5.7 출시, -preview 아님 주의)
judge_client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])

print("환경설정 완료.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google AI Studio API Key 입력 (임베딩 조회 + 채점 공용): ··········
환경설정 완료.


In [ ]:
# ============================================================
# ===== 1. esg_query_interface.ipynb 함수 재사용 (원문 그대로) =====
# ============================================================
kiwi = Kiwi()
RRF_K = 60
KIWI_KEEP_TAGS = {"NNG", "NNP", "NNB", "VV", "VA", "SL", "SN", "SH"}


def kiwi_tokenize(text: str):
    return [t.form for t in kiwi.tokenize(text) if t.tag in KIWI_KEEP_TAGS]


def restore_local_from_drive_backup():
    if os.path.exists(CONFIG["persist_dir"]):
        print("로컬 인덱스 이미 존재 (복원 스킵)")
        return
    if os.path.exists(CONFIG["drive_backup_dir"]):
        print("Drive 백업 -> 로컬 복원 중")
        shutil.copytree(CONFIG["drive_backup_dir"], CONFIG["persist_dir"])
    else:
        raise FileNotFoundError("Drive 백업이 없음 - 관리자에게 인덱싱 진행 상황 확인할 것")


class FilteredHybridRetriever:
    """메인 파이프라인과 동일한 구현 (BM25를 필터로 좁혀진 부분집합에서만 생성)."""

    def __init__(self, vectorstore, tokenize_func, k=10, weights=(0.5, 0.5), max_cache_entries=30):
        self.vectorstore = vectorstore
        self.tokenize_func = tokenize_func
        self.k = k
        self.weights = weights
        self._bm25_cache: dict[str, object] = {}
        self._max_cache_entries = max_cache_entries

    def _get_bm25_retriever(self, filter_dict):
        sig = json.dumps(filter_dict, sort_keys=True, ensure_ascii=False)
        if sig in self._bm25_cache:
            return self._bm25_cache[sig]
        raw = self.vectorstore.get(where=filter_dict if filter_dict else None,
                                    include=["documents", "metadatas"])
        docs = [Document(page_content=d, metadata=m) for d, m in zip(raw["documents"], raw["metadatas"])]
        if not docs:
            self._set_cache(sig, None)
            return None
        retriever = BM25Retriever.from_documents(docs, preprocess_func=self.tokenize_func)
        retriever.k = self.k
        self._set_cache(sig, retriever)
        return retriever

    def _set_cache(self, sig, value):
        if len(self._bm25_cache) >= self._max_cache_entries:
            oldest_key = next(iter(self._bm25_cache))
            del self._bm25_cache[oldest_key]
        self._bm25_cache[sig] = value

    def invoke(self, query: str, filter: dict | None = None, top_n: int | None = None,
               query_embedding: list[float] | None = None):
        top_n = top_n or self.k
        if query_embedding is not None:
            dense_docs = self.vectorstore.similarity_search_by_vector(
                query_embedding, k=self.k, filter=filter
            )
        else:
            dense_docs = self.vectorstore.similarity_search(query, k=self.k, filter=filter)
        bm25_retriever = self._get_bm25_retriever(filter)
        bm25_docs = bm25_retriever.invoke(query) if bm25_retriever else []

        scores: dict[str, float] = {}
        lookup: dict[str, Document] = {}
        for rank, doc in enumerate(dense_docs):
            key = doc.metadata["chunk_id"]
            scores[key] = scores.get(key, 0.0) + self.weights[0] * (1.0 / (RRF_K + rank))
            lookup[key] = doc
        for rank, doc in enumerate(bm25_docs):
            key = doc.metadata["chunk_id"]
            scores[key] = scores.get(key, 0.0) + self.weights[1] * (1.0 / (RRF_K + rank))
            lookup[key] = doc

        ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
        return [lookup[k] for k, _ in ranked[:top_n]]

    def clear_cache(self):
        self._bm25_cache.clear()


restore_local_from_drive_backup()
embeddings = GoogleGenerativeAIEmbeddings(model=CONFIG["embedding_model"])
vectorstore = Chroma(persist_directory=CONFIG["persist_dir"], embedding_function=embeddings)
hybrid_retriever = FilteredHybridRetriever(
    vectorstore=vectorstore, tokenize_func=kiwi_tokenize,
    k=CONFIG["candidate_k"], weights=CONFIG["hybrid_weights"],
)
print("Chroma DB + 하이브리드 리트리버 로드 완료")


def load_rubric(path: str, sheet_name: str = "01_ESG통합진단마스터") -> tuple[pd.DataFrame, int]:
    """반환: (데이터프레임, 실제 엑셀 헤더 행번호[1-based])
    분리본(E/S/G+P+반도체특화)은 1행에 설명 문구가 있어 헤더가 2행,
    전체 통합본은 헤더가 1행입니다. 어느 쪽인지 자동 감지해서 함께 반환합니다.
    """
    df = pd.read_excel(path, sheet_name=sheet_name)
    header_row = 1
    if "분류코드" not in df.columns:
        df = pd.read_excel(path, sheet_name=sheet_name, header=1)
        header_row = 2
    df.columns = [str(c).strip() for c in df.columns]
    for col in ("진단항목명", "실사 진단 질문"):
        if col in df.columns:
            df[col] = df[col].fillna("")
    return df, header_row


rubric_df, RUBRIC_HEADER_ROW = load_rubric(CONFIG["rubric_xlsx"])
rubric_by_code = rubric_df.set_index("분류코드").to_dict(orient="index")
print(f"항목 {len(rubric_df)}개 로드 완료 (헤더 위치: {RUBRIC_HEADER_ROW}행)")
print(rubric_df["영역"].value_counts())


def load_score_lookup(path: str, sheet_name: str = "02_점수산정참고표") -> dict:
    """02번 시트(단계별 판단기준+배점)를 분류코드별로 그룹핑해서 반환.
    - {분류코드: [{"단계번호":1, "배점":0, "판단기준":"..."}, ...]}
    - X열(자사취득점수)을 엑셀 수식 대신 파이썬에서 직접 계산하는 데 사용
    - 프롬프트에 "단계별 상세 기준"을 명시적으로 제공하는 데도 사용
    """
    df = pd.read_excel(path, sheet_name=sheet_name)
    if "분류코드" not in df.columns:
        df = pd.read_excel(path, sheet_name=sheet_name, header=1)
    df.columns = [str(c).strip() for c in df.columns]

    lookup: dict[str, list[dict]] = {}
    for _, row in df.iterrows():
        code = row.get("분류코드")
        if pd.isna(code):
            continue
        lookup.setdefault(code, []).append({
            "단계번호": row.get("단계번호"),
            "배점": row.get("배점"),
            "판단기준": row.get("단계별 판단기준(K-ESG v2.0 원문)", ""),
        })
    return lookup


score_lookup_by_code = load_score_lookup(CONFIG["rubric_xlsx"])
print(f"02번 시트(단계별 배점표) {len(score_lookup_by_code)}개 항목 로드 완료")


RISKY_DOCS: dict[tuple, float] = {}
RISKY_COMPANIES: set[str] = set()

if os.path.exists(CONFIG["extraction_diag_csv"]):
    _diag = pd.read_csv(CONFIG["extraction_diag_csv"], dtype={"company_code": str})
    _risky = _diag[_diag["issue_ratio_after"] > 0.3]
    for _, r in _risky.iterrows():
        _code = str(r["company_code"]).zfill(6)
        RISKY_DOCS[(_code, int(r["year"]), r["doc_type"])] = r["issue_ratio_after"]
        RISKY_COMPANIES.add(_code)
    if RISKY_DOCS:
        print(f"⚠️ 텍스트 추출 실패율이 높은 문서 {len(RISKY_DOCS)}건 존재")
else:
    print("extraction_diagnostics.csv 없음 - 위험도 체크 생략")


CHECKPOINT_PATH = f"{DRIVE_ROOT}/checkpoint_completed.json"
REQUIRED_DOC_TYPES = {"business_report", "governance_report", "sustainability_report"}


def list_available_companies(detail: bool = False) -> pd.DataFrame:
    if not os.path.exists(CHECKPOINT_PATH):
        print("체크포인트 파일 없음")
        return pd.DataFrame()
    with open(CHECKPOINT_PATH, encoding="utf-8") as f:
        checkpoint = json.load(f)
    rows = []
    for doc_id in checkpoint:
        parts = doc_id.split("_")
        rows.append({"company_code": parts[0], "year": parts[1], "doc_type": "_".join(parts[2:])})
    df = pd.DataFrame(rows)
    year_level = (
        df.groupby(["company_code", "year"])["doc_type"]
        .apply(lambda s: sorted(set(s))).reset_index()
    )
    year_level["연도_3종_완비"] = year_level["doc_type"].apply(lambda t: set(t) == REQUIRED_DOC_TYPES)
    if detail:
        year_level["위험도_경고"] = year_level["company_code"].isin(RISKY_COMPANIES)
        return year_level.sort_values(["company_code", "year"])
    return year_level


_query_embedding_cache: dict[str, list[float]] = {}


def get_evidence(company_code: str, year: int, item_code: str, top_n: int = 10,
                  doc_type: str | None = None) -> list[dict]:
    company_code = str(company_code).zfill(6)
    if item_code not in rubric_by_code:
        raise KeyError(f"채점기준표에 없는 분류코드: {item_code}")
    item = rubric_by_code[item_code]
    query = f"{item['진단항목명']} {item['실사 진단 질문']}"
    conditions = [{"company_code": company_code}, {"year": year}]
    if doc_type:
        conditions.append({"doc_type": doc_type})
    filter_dict = {"$and": conditions}
    if item_code not in _query_embedding_cache:
        _query_embedding_cache[item_code] = embeddings.embed_query(query)
    query_embedding = _query_embedding_cache[item_code]
    results = hybrid_retriever.invoke(query, filter=filter_dict, top_n=top_n, query_embedding=query_embedding)
    if not results:
        print(f"⚠️ {company_code}/{year}년 근거 0건 - list_available_companies(detail=True)로 확인할 것")
    return [
        {"doc_type": d.metadata.get("doc_type"), "page": d.metadata.get("page"),
         "section_title": d.metadata.get("section_title"), "content": d.page_content}
        for d in results
    ]


def build_scoring_prompt(company_code: str, year: int, item_code: str, evidence: list[dict]) -> str:
    """팀 공용 프롬프트 뼈대 - 내용 절대 수정하지 않음 (일관성 유지)."""
    item = rubric_by_code[item_code]
    evidence_text = "\n\n".join(
        f"[근거 {i+1}] ({e['doc_type']} p.{e['page']}, 섹션: {e['section_title'] or '미상'})\n{e['content']}"
        for i, e in enumerate(evidence)
    ) or "(검색된 근거 없음)"
    return f"""다음은 K-ESG 진단항목 채점을 위한 정보입니다.

[대상]
- 기업코드: {company_code} / 채점 연도: {year} (해당 연도 보고서 기준으로만 채점할 것)

[항목 정보]
- 분류코드: {item_code}
- 영역/범주: {item['영역']} / {item['범주']}
- 진단항목명: {item['진단항목명']}
- 항목설명: {item['항목설명']}
- 채점유형: {item['채점유형']} (답변형식: {item['답변형식']})
- 점검기준: {item['점검기준 상세(K-ESG v2.0 원문)']}
- 실사 진단 질문: {item['실사 진단 질문']}

[검색된 근거]
{evidence_text}

위 근거만을 바탕으로 이 항목을 채점하세요. 근거가 불충분하면 "근거부족"이라고 답하세요.
"""

로컬 인덱스 이미 존재 (복원 스킵)
Chroma DB + 하이브리드 리트리버 로드 완료
항목 91개 로드 완료 (헤더 위치: 1행)
영역
사회      34
환경      32
지배구조    20
정보공시     5
Name: count, dtype: int64
02번 시트(단계별 배점표) 74개 항목 로드 완료
⚠️ 텍스트 추출 실패율이 높은 문서 2건 존재


In [ ]:
# ============================================================
# ===== 2. 채점 대상 기업 + 연도 설정 (여기만 채워 넣으면 됨) =====
# ============================================================
# ⚠️ 파일럿: SK하이닉스(000660) 1개사로 91개 전체 항목 채점 검증
TARGET_COMPANIES = [
    {"company_code": "000660", "company_name": "SK하이닉스", "years": [2022, 2023, 2024, 2025]},
]
IS_C261 = True  # SK하이닉스는 반도체(KSIC 26111, 메모리) 제조사 - 반도체 특화 6개 항목도 채점 대상
# ⚠️ 위 연도는 일단 4개년 전체로 넣어뒀습니다. 아래 list_available_companies() 출력의
#    "완비_연도목록" 컬럼을 보고, 실제로 3종 완비 안 된 연도가 있으면 여기서 빼고 실행하세요.
#    (완비 안 된 연도를 그대로 두면, 그 연도만 근거 0건으로 나오고 최하단계 처리됩니다)

print("\n현재 준비된(3종 완비) 회사/연도 목록:")
ready_df = list_available_companies()
print(ready_df)

# ---- 안전장치: 방금 채워 넣은 TARGET_COMPANIES가 실제로 준비됐는지 자동 점검 ----
_ready_codes = set(ready_df["company_code"]) if not ready_df.empty else set()

for company in TARGET_COMPANIES:
    code = company["company_code"]
    if code not in _ready_codes:
        print(f"⚠️ 경고: {code}는 아직 인덱싱이 안 됐거나 3종 완비 연도가 없습니다. "
              f"list_available_companies(detail=True)로 먼저 확인하세요.")
    if code in RISKY_COMPANIES:
        print(f"⚠️ 참고: {code}는 텍스트 추출 실패율이 높은 문서가 포함된 회사입니다 (결과 신뢰도 낮을 수 있음).")


현재 준비된(3종 완비) 회사/연도 목록:
    company_code  year                                           doc_type  \
0         000500  2022           [business_report, sustainability_report]   
1         000500  2023           [business_report, sustainability_report]   
2         000500  2024  [business_report, governance_report, sustainab...   
3         000500  2025  [business_report, governance_report, sustainab...   
4         000660  2022  [business_report, governance_report, sustainab...   
..           ...   ...                                                ...   
172       373220  2024  [business_report, governance_report, sustainab...   
173       373220  2025  [business_report, governance_report, sustainab...   
174       393890  2024                                  [business_report]   
175       393890  2025                            [sustainability_report]   
176       450080  2025  [business_report, governance_report, sustainab...   

     연도_3종_완비  
0       False  
1       False  
2 

In [ ]:
# ============================================================
# ===== 3. Gemini 판정 함수 (채점유형별 출력형식 지시문 + 계산 로직) =====
# ============================================================
# [260904 QA 수정] 최호연·김민주님의 91문항 수기 검수에서 발견된 "출처 문서 혼동"
# (P-1-1/P-1-2/P-1-3/S-8-2/S-9-1 등: 존재하지 않는 페이지/문서로 근거가 표기됨)와
# "복합가중형 계산 오류"(G-3-3: 4개년 내내 1.3점 고정, 2024·2025년엔 전자투표제 도입
# 근거가 있는데도 그대로)를 이 노트북의 실제 로직에서 재현·확인하고 아래 2곳을 고쳤음:
#   1) locate_citation(): LLM이 뽑은 "핵심근거문구"가 실제 근거 텍스트 어디에도
#      문자열로 매칭되지 않을 때, 예전에는 evidence[0](=하이브리드 검색 1순위 청크)의
#      문서/페이지로 임의 귀속시켰다. 근거가 여러 문서(사업/지배구조/지속가능경영보고서)
#      에 걸쳐 있을 때 이게 엉뚱한 문서/페이지를 "확정된 근거"처럼 적어버리는 원인이었다
#      (실제 데이터에서 "(줄 위치 특정 실패)" 태그가 다수 발견됨 = 이 폴백이 실제로 자주
#      발동하고 있었다는 증거). 이제는 근거가 1건뿐일 때만 그 문서로 귀속하고, 여러 건이면
#      문서/페이지를 확정하지 않고 "자동 특정 실패"로만 표시 + needs_review=True.
#   2) 복합가중형: 세부요소 점수를 "가중치 없이 단순 평균"하던 것을, LLM이 세부요소별
#      가중치(0~100, 합 100)를 함께 답하게 해서 가중평균으로 바꿈. LLM이 가중치를
#      안 주면(구버전 호환) 기존과 동일하게 균등 평균으로 자동 폴백 — 있던 동작을
#      절대 더 나쁘게 만들지 않음.
# ============================================================
def _safe_int(value, default: int = 0) -> int:
    """LLM이 <정수>로 답하라고 지시받아도 "0"(문자열), "없음", "2건" 등
    형식을 안 지키는 경우가 있어, 어떤 값이 와도 안전하게 정수로 변환합니다.
    - 이미 숫자면 그대로 int 변환 (단, NaN은 빈 값으로 간주 - 엑셀 빈 셀을 pandas가
      읽으면 NaN이 되는데, int(nan)은 예외를 던지므로 반드시 먼저 걸러야 함)
    - 문자열이면 그 안에서 숫자만 추출 (예: "2건" → 2)
    - 숫자를 전혀 못 찾으면 default(기본 0) 반환
    """
    if value is None:
        return default
    if isinstance(value, float) and math.isnan(value):
        return default
    if isinstance(value, (int, float)):
        return int(value)
    if isinstance(value, str):
        match = re.search(r"-?\d+", value)
        if match:
            return int(match.group())
    return default


def gemini_judge(prompt: str, schema_keys: list[str]) -> dict:
    """LLM 응답을 dict로 파싱. 두 가지 실패 케이스를 모두 방어합니다:
    1) JSON 자체가 깨진 경우 (JSONDecodeError)
    2) JSON 파싱은 성공했지만 dict가 아닌 경우 - 예: build_scoring_prompt()의
       "근거가 불충분하면 '근거부족'이라고 답하세요" 지시를 LLM이 그대로 따라서
       {"단계번호": ...} 대신 그냥 문자열 "근거부족"만 반환하는 경우
       (json.loads('"근거부족"')는 에러 없이 성공하지만 결과가 str이라 .get() 호출 시 터짐)
    두 경우 모두 "판단 불가"로 간주해 schema_keys를 전부 None으로 채운 dict를 돌려주고,
    호출부(score_item)에서 이걸 감지해 최하단계/0건 처리 + AI검토필요 플래그를 답니다.
    """
    response = judge_client.models.generate_content(
        model=JUDGE_MODEL, contents=prompt,
        config=genai_types.GenerateContentConfig(temperature=0, response_mime_type="application/json"),
    )
    try:
        result = json.loads(response.text)
    except (json.JSONDecodeError, TypeError):
        result = None

    if not isinstance(result, dict):
        return {k: None for k in schema_keys}
    return result


def _lookup_score(item_code: str, stage: int) -> int:
    """02번 시트를 참조해 '단계번호'를 실제 배점으로 변환합니다.
    (기존에는 엑셀 SUMPRODUCT 수식이 이 역할을 했으나, openpyxl은 수식을 재계산하지
    않아 X열이 계속 공란으로 남는 문제가 있었습니다. 이제 파이썬에서 직접 계산합니다.)
    """
    rows = score_lookup_by_code.get(item_code, [])
    for r in rows:
        if _safe_int(r["단계번호"]) == stage:
            return _safe_int(r["배점"])
    return 0  # 매칭되는 단계를 못 찾으면 0점 (방어적 기본값)


def _get_max_stage(item_code: str, item: dict) -> int:
    """이 항목이 몇 단계까지 있는지(엑셀 데이터 검증 상한과 반드시 일치해야 함) 확인합니다.
    - 02번 시트에 실제로 정의된 단계번호 중 최댓값을 우선 사용 (가장 정확)
    - 없으면 01번 시트의 '최대단계수' 컬럼값으로 대체
    - 그마저도 없으면 방어적으로 5 (K-ESG 표준 단계 수)
    """
    rows = score_lookup_by_code.get(item_code, [])
    if rows:
        return max(_safe_int(r["단계번호"], default=1) for r in rows)
    fallback = _safe_int(item.get("최대단계수"), default=0)
    return fallback if fallback > 0 else 5


DOC_TYPE_KOR = {
    "business_report": "사업보고서",
    "governance_report": "지배구조보고서",
    "sustainability_report": "지속가능경영보고서",
}


def locate_citation(snippet: str | None, evidence: list[dict]) -> tuple[str, bool]:
    """LLM이 뽑아낸 '핵심 근거 문구'가 실제로 어느 근거(문서/페이지/몇 번째 줄)에서
    나왔는지 문자열 매칭으로 정확히 찾아서, 요청하신 인용 형식으로 변환합니다.
    형식: [문서명] Np. 문장(N번째 줄)

    LLM에게 직접 "몇 번째 줄이야?"라고 물어보지 않는 이유: LLM은 정확한 줄 세기에
    약해서 신뢰할 수 없습니다. 대신 LLM은 "어떤 문구인지"만 찾게 하고, 그 문구가
    실제로 검색된 텍스트의 몇 번째 줄에 있는지는 파이썬이 문자열 매칭으로 확인합니다.

    반환: (표시할 인용 문자열, 실제로 위치를 확인했는지 여부)

    [260904 QA 수정] 문자열 매칭에 실패했을 때, 예전에는 evidence[0](하이브리드 검색
    1순위로 뽑힌 청크)의 문서/페이지로 그냥 귀속시켰다. 근거가 사업/지배구조/지속가능
    경영보고서 등 여러 문서에 걸쳐 섞여 들어온 상태에서는 evidence[0]이 LLM이 실제로
    인용한 문구와 무관한 문서일 수 있어, "존재하지 않는 페이지"류 오류(P-1-1/P-1-2 등
    검수에서 지적된 패턴)의 직접적인 원인이었다. 근거가 1건뿐일 때만 그 문서로 안전하게
    귀속하고, 여러 건이면 문서/페이지를 단정하지 않는다 (호출부에서 needs_review=True).
    """
    if not snippet or not snippet.strip():
        return "(근거 문구 없음)", False
    snippet_clean = snippet.strip()
    probe = snippet_clean[:20]  # 앞부분 일부만으로 매칭 (LLM이 문구를 약간 다르게 옮겨 적어도 매칭되게)

    for e in evidence:
        lines = (e.get("content") or "").split("\n")
        for line_no, line in enumerate(lines, 1):
            if probe and (probe in line or line.strip()[:20] in snippet_clean):
                doc_name = DOC_TYPE_KOR.get(e.get("doc_type"), e.get("doc_type", "문서"))
                return f"[{doc_name}] {e.get('page')}p. {snippet_clean}({line_no}번째 줄)", True

    # [260904 QA 수정] 정확한 줄 위치를 못 찾은 경우:
    #  - 근거가 1건뿐이면 그 문서/페이지는 확실하므로 그대로 표기 (줄 번호만 불명확)
    #  - 근거가 여러 건이면 어느 문서/페이지에서 나온 문구인지 알 수 없으므로
    #    특정 문서/페이지를 적지 않는다 (예전처럼 evidence[0]로 임의 귀속하지 않음)
    if len(evidence) == 1:
        doc_name = DOC_TYPE_KOR.get(evidence[0].get("doc_type"), evidence[0].get("doc_type", "문서"))
        return f"[{doc_name}] {evidence[0].get('page')}p. {snippet_clean}(줄 위치 특정 실패)", False
    if evidence:
        return (
            f"(출처 자동 특정 실패 - 근거 {len(evidence)}건 중 어느 문서/페이지인지 불명확: "
            f"\"{snippet_clean}\")"
        ), False
    return snippet_clean, False


def score_item(company_code: str, year: int, item_code: str) -> dict:
    item = rubric_by_code[item_code]
    scoring_type = item["채점유형"]
    improvement_guide = str(item.get("개선 가이드") or "")[:500]  # 참고자료 - 토큰 절약 위해 500자 제한

    evidence = get_evidence(company_code, year, item_code, top_n=5)
    base_prompt = build_scoring_prompt(company_code, year, item_code, evidence)

    if not evidence:
        base = {"evidence_pages": [], "needs_review": True, "review_reason": "근거 미검색",
                "judge_reasoning": "검색된 근거 없음", "sufficiency": "불충분"}
        if scoring_type in ("단계형", "선택형"):
            base["stage"] = 1
            base["score"] = _lookup_score(item_code, 1)
        elif scoring_type == "감점형":
            base["violations"] = {"유형1": 0, "유형2": 0, "유형3": 0}
            base["score"] = 100
        else:
            base["score"] = 0
        return base

    evidence_pages = sorted({e["page"] for e in evidence})

    # ---- 참고자료 추가: 02번 시트 단계별 상세기준 + 개선 가이드 ----
    # (build_scoring_prompt() 자체는 수정하지 않고, 그 뒤에 추가 컨텍스트로 덧붙임)
    context_addendum = ""
    max_stage = _get_max_stage(item_code, item)
    if scoring_type in ("단계형", "선택형") and item_code in score_lookup_by_code:
        stage_detail = "\n".join(
            f"- {r['단계번호']}단계({r['배점']}점): {r['판단기준']}"
            for r in score_lookup_by_code[item_code]
        )
        context_addendum += f"\n[단계별 상세 판단기준]\n{stage_detail}\n"
    if improvement_guide:
        context_addendum += f"\n[참고: 개선 가이드 - 상위 단계가 요구하는 조건 파악에 참고]\n{improvement_guide}\n"

    SUFFICIENCY_SPEC = (
        '- "근거충분성": 검색된 근거가 이 항목을 판단하기에 충분한지 "충분"/"부분적"/"불충분" '
        "중 하나로 평가 (근거가 간접적이거나 일부만 확인되면 '부분적', 거의 무관하면 '불충분')\n"
    )

    if scoring_type in ("단계형", "선택형"):
        prompt = (
            base_prompt + context_addendum +
            '\n반드시 아래 JSON 형식으로만 답하세요.\n'
            f'- "단계번호": 이 항목은 **1단계부터 {max_stage}단계까지만 존재**합니다. '
            f"반드시 1~{max_stage} 사이의 **정수 순서 번호(인덱스)**로만 답하세요. "
            "이것은 점수나 백분율이 아니라 '몇 번째 단계인지'를 나타내는 순번입니다 "
            f"(예: 0이나 100처럼 범위를 벗어난 값, 배점 자체를 답하면 안 됩니다. 반드시 1~{max_stage} 중 하나).\n"
            '- "핵심근거문구": 근거 자료 원문에 실제로 존재하는 문장을 그대로(요약/의역 금지) '
            '짧게 발췌 (나중에 원문에서 위치를 찾아야 하므로 정확히 옮겨 적을 것)\n'
            '- "판단설명": 그 근거가 왜 이 단계에 해당하는지 1문장 설명 '
            '(예: "탄소중립 정량 목표(43% 감축)를 명시하여 5단계 요건 충족함", '
            '"정성적 목표만 있고 정량 목표가 없어 4단계에 그침" 등 충족/미충족 여부를 명확히 밝힐 것)\n'
            + SUFFICIENCY_SPEC +
            f'{{"단계번호": <1~{max_stage} 사이 정수>, "핵심근거문구": "<원문 그대로 발췌>", '
            '"판단설명": "<충족/미충족 이유 1문장>", "근거충분성": "<충분|부분적|불충분>"}'
        )
        result = gemini_judge(prompt, ["단계번호", "핵심근거문구", "판단설명", "근거충분성"])
        판단_불가 = result.get("단계번호") is None
        raw_stage = _safe_int(result.get("단계번호"), default=1)

        # ---- 범위 방어: 엑셀 데이터 검증(1~max_stage)을 벗어난 값은 강제 보정 ----
        out_of_range = not (1 <= raw_stage <= max_stage)
        stage = min(max(raw_stage, 1), max_stage) if out_of_range else raw_stage

        score = _lookup_score(item_code, stage)
        citation, located = locate_citation(result.get("핵심근거문구"), evidence)
        explanation = result.get("판단설명") or "(판단설명 없음)"
        sufficiency = result.get("근거충분성") or "불충분"

        insufficient = sufficiency == "불충분"
        needs_review = 판단_불가 or out_of_range or not located or insufficient
        reasons = []
        if 판단_불가:
            reasons.append("LLM이 근거부족으로 판단(최하단계 처리)")
        if out_of_range:
            reasons.append(f"LLM이 유효범위(1~{max_stage}) 밖의 값({raw_stage})을 답해 {stage}로 보정함")
        if not located and not 판단_불가:
            reasons.append("근거 출처(문서/페이지) 자동 특정 실패 - 원문 대조 필요")
        if insufficient and not 판단_불가:
            # [260904 QA 수정] LLM이 자체적으로 "근거충분성: 불충분"이라고 답해도 지금까지는
            # 비고란에 라벨만 남기고 needs_review에는 전혀 반영되지 않았다 (E-5-2 2025년치처럼
            # 근거충분성이 불충분인데도 검토 플래그 없이 그대로 확정되는 사례의 직접 원인).
            reasons.append("LLM 자체 평가 근거충분성=불충분 - 재검토 필요")
        review_reason = " · ".join(reasons) if reasons else None

        return {"stage": stage, "score": score, "evidence_pages": evidence_pages,
                "judge_reasoning": f"{citation} → {explanation}", "sufficiency": sufficiency,
                "needs_review": needs_review, "review_reason": review_reason}

    elif scoring_type == "감점형":
        prompt = (
            base_prompt + context_addendum +
            '\n반드시 아래 JSON 형식으로만 답하세요.\n'
            '- "_해당여부": 그 유형에 해당하는 위반이 **하나라도 있으면 1, 전혀 없으면 0** '
            "(위반이 여러 건이어도 그 유형은 1로만 표시 - 건수를 세지 마세요. "
            '"중복 적용 가능"은 "유형1과 유형2가 동시에 있으면 둘 다 1"이라는 뜻이지, '
            "같은 유형이 여러 번 있다고 더 깎인다는 뜻이 아닙니다)\n"
            '해당여부가 1이면 반드시 "_근거문구"에 위반을 확인한 근거자료 원문 문장을 '
            '그대로 발췌하고(요약/의역 금지), "_설명"에 이유를 1문장으로 쓰세요. '
            "해당 없으면 근거문구/설명은 null로 두세요.\n"
            + SUFFICIENCY_SPEC +
            '{"유형1_해당여부": <0 또는 1>, "유형1_근거문구": "<원문 발췌 또는 null>", "유형1_설명": "<이유 또는 null>", '
            '"유형2_해당여부": <0 또는 1>, "유형2_근거문구": "<원문 발췌 또는 null>", "유형2_설명": "<이유 또는 null>", '
            '"유형3_해당여부": <0 또는 1>, "유형3_근거문구": "<원문 발췌 또는 null>", "유형3_설명": "<이유 또는 null>", '
            '"근거충분성": "<충분|부분적|불충분>"}'
        )
        result = gemini_judge(prompt, ["유형1_해당여부", "유형2_해당여부", "유형3_해당여부", "근거충분성"])
        판단_불가 = all(result.get(k) is None for k in ("유형1_해당여부", "유형2_해당여부", "유형3_해당여부"))

        # 방어: LLM이 여전히 "건수"처럼 2 이상을 답해도 0/1로 강제 클램핑
        def _to_flag(v):
            n = _safe_int(v)
            return 1 if n >= 1 else 0

        violations = {
            "유형1": _to_flag(result.get("유형1_해당여부")),
            "유형2": _to_flag(result.get("유형2_해당여부")),
            "유형3": _to_flag(result.get("유형3_해당여부")),
        }
        score = max(0, 100 - (violations["유형1"] * 50 + violations["유형2"] * 30 + violations["유형3"] * 10))
        citation_parts = []
        any_unlocated = False
        for i in (1, 2, 3):
            if violations[f"유형{i}"] > 0:
                snippet = result.get(f"유형{i}_근거문구")
                explanation = result.get(f"유형{i}_설명") or "(설명 없음)"
                citation, located = locate_citation(snippet, evidence)
                if not located:
                    any_unlocated = True
                citation_parts.append(f"[유형{i}] {citation} → {explanation}")
        judge_reasoning = " / ".join(citation_parts) if citation_parts else "위반사항 없음"
        sufficiency = result.get("근거충분성") or "불충분"

        if 판단_불가:
            return {"violations": {"유형1": 0, "유형2": 0, "유형3": 0}, "score": 100, "evidence_pages": evidence_pages,
                    "judge_reasoning": "LLM 응답 파싱 실패", "sufficiency": "불충분",
                    "needs_review": True, "review_reason": "LLM이 근거부족으로 판단(0건 처리)"}
        insufficient = sufficiency == "불충분"
        needs_review = any(v > 0 for v in violations.values()) or any_unlocated or insufficient
        reasons = []
        if any(violations.values()):
            reasons.append("위반 발견 - 재확인 권장")
        if any_unlocated:
            reasons.append("근거 출처(문서/페이지) 자동 특정 실패 - 원문 대조 필요")
        if insufficient:
            reasons.append("LLM 자체 평가 근거충분성=불충분 - 재검토 필요")
        return {"violations": violations, "score": score, "evidence_pages": evidence_pages,
                "judge_reasoning": judge_reasoning, "sufficiency": sufficiency,
                "needs_review": needs_review,
                "review_reason": " · ".join(reasons) if reasons else None}

    else:  # 복합가중형
        # [260904 QA 수정] "가중치"를 LLM에게 함께 요청해서, 세부요소 점수를 균등 평균이
        # 아니라 가중평균으로 계산한다 (G-3-3처럼 특정 세부요소만 비중이 높은 항목에서
        # 균등 평균이 실제 배점 구조와 다른 결과를 내는 문제 대응). LLM이 가중치를 안 주면
        # (예: 구버전 프롬프트 캐시, 파싱 실패) 기존과 동일하게 균등 평균으로 폴백한다 —
        # 즉 이 수정은 상황을 더 나쁘게 만들 수 없고, 가중치가 확보될 때만 개선된다.
        prompt = (
            base_prompt + context_addendum +
            '\n이 항목은 여러 세부요소로 나뉘어 있고, 각 세부요소마다 자체적인 단계(1~N단계) '
            "기준이 있습니다. 세부요소별로 아래를 답하세요.\n"
            '- "적용단계": 근거를 보고 그 세부요소가 몇 단계에 해당하는지(1부터 시작하는 정수)\n'
            '- "최대단계": 그 세부요소가 원문 기준 몇 단계까지 있는지(예: 1~3단계면 3, 1~2단계면 2)\n'
            '- "가중치": 위 [점검기준]에 명시된 배점/비중 구조를 참고해서, 이 세부요소가 '
            "전체 100점 중 차지하는 비중을 정수(0~100)로 답하세요. 모든 세부요소의 가중치 합은 "
            "반드시 100이 되어야 합니다. 점검기준에 비중이 명시되어 있지 않고 세부요소들이 "
            "동등하게 취급되어야 한다면 균등하게 나눠서(예: 3개면 각각 약 33~34) 답하세요.\n"
            "점수는 절대로 단계번호를 그대로 쓰지 마세요 - 각 세부요소의 단계별 점수 환산은 "
            "코드가 자동으로 (적용단계-1)/(최대단계-1)×100 공식으로 계산하고, 최종 점수는 "
            "그 값들을 가중치대로 가중평균합니다. 당신은 적용단계/최대단계/가중치 숫자만 "
            '정확히 답하면 됩니다.\n'
            '"근거문구"는 근거자료 원문을 그대로 발췌하고(요약/의역 금지), "설명"에는 왜 그 '
            "단계인지 1문장으로 쓰세요.\n"
            + SUFFICIENCY_SPEC +
            '{"세부요소": [{"요소명": "...", "적용단계": <정수>, "최대단계": <정수>, '
            '"가중치": <0~100 정수>, "근거문구": "<원문 발췌>", "설명": "<이유>"}, ...], '
            '"근거충분성": "<충분|부분적|불충분>"}'
        )
        result = gemini_judge(prompt, ["세부요소", "근거충분성"])
        sub = result.get("세부요소") or []

        scored = []  # (sub_score, weight, element_dict)
        for e in sub:
            if not isinstance(e, dict):
                continue
            max_st = max(_safe_int(e.get("최대단계"), default=2), 2)  # 최소 2단계(0/100) 가정, 방어적 하한
            applied = min(max(_safe_int(e.get("적용단계"), default=1), 1), max_st)  # 1~max_st로 클램핑
            sub_score = (applied - 1) / (max_st - 1) * 100
            weight = _safe_int(e.get("가중치"), default=0)
            scored.append((sub_score, weight, e))

        total_weight = sum(w for _, w, _ in scored)
        any_unlocated = False
        if total_weight > 0:
            composite_score = round(sum(s * w for s, w, _ in scored) / total_weight, 1)
        elif scored:
            # 가중치를 못 받은 경우(구버전 호환) - 기존 동작 그대로 균등 평균
            composite_score = round(sum(s for s, _, _ in scored) / len(scored), 1)
        else:
            composite_score = 0

        reasoning_parts = []
        for s, w, e in scored:
            if not e.get("근거문구"):
                continue
            citation, located = locate_citation(e.get("근거문구"), evidence)
            if not located:
                any_unlocated = True
            reasoning_parts.append(f"[{e.get('요소명', '')}·가중치{w}] {citation} → {e.get('설명', '(설명 없음)')}")
        reasoning = " / ".join(reasoning_parts) or "(세부 근거 없음)"
        sufficiency = result.get("근거충분성") or "부분적"
        review_reason = "복합가중형 - 표본검증 권장"
        if any_unlocated:
            review_reason += " · 근거 출처 자동 특정 실패 일부 포함"
        return {"score": composite_score, "evidence_pages": evidence_pages,
                "judge_reasoning": reasoning, "sufficiency": sufficiency,
                "needs_review": True, "review_reason": review_reason}


In [ ]:
# ============================================================
# ===== 4. 채점 실행 (5개사 x 각자 연도 목록 순회) =====
# ============================================================
COL = {"충족단계": "T", "위반1": "U", "위반2": "V", "위반3": "W", "점수": "X", "비고": "Z"}


def find_excel_row(item_code: str) -> int:
    """rubric_df 상의 위치를 실제 엑셀 행번호로 변환.
    헤더가 RUBRIC_HEADER_ROW행에 있으므로, 데이터는 그다음 행부터 시작합니다.
    """
    pos = rubric_df.index[rubric_df["분류코드"] == item_code][0]  # 0-based 위치
    return pos + RUBRIC_HEADER_ROW + 1


RUBRIC_OUTPUT_ROOT = "/content/drive/MyDrive/ESG 팀프로젝트/2. ESG 채점 세트/채점 모델/채점모델_안지예"

item_codes = rubric_df["분류코드"].tolist()

for company in TARGET_COMPANIES:
    code, name = company["company_code"], company["company_name"]
    for year in company["years"]:
        print(f"\n{'='*20} [{name}({code}) {year}년] 전체 91개 항목 채점 시작 {'='*20}")

        wb_out = openpyxl.load_workbook(CONFIG["rubric_xlsx"])  # 매번 원본 템플릿에서 새로 로드
        ws_out = wb_out["01_ESG통합진단마스터"]

        for i, item_code in enumerate(item_codes, 1):
            item_name = rubric_by_code[item_code]["진단항목명"]
            print(f"  [{i}/{len(item_codes)}] {item_code} {item_name} 채점 중...")
            r = find_excel_row(item_code)
            scoring_type = rubric_by_code[item_code]["채점유형"]

            # ---- 안전망: 아직 못 본 종류의 에러가 터져도 이 항목만 건너뛰고 계속 진행 ----
            # (단, gemini_judge/score_item 안에서 이미 처리되는 known 케이스는 여기까지
            #  안 옴 - 이 except는 "완전히 예상 못 한" 에러 전용 최후 방어선)
            try:
                result = score_item(code, year, item_code)
            except Exception as e:
                print(f"    ⚠️ 예상치 못한 에러로 이 항목 건너뜀: {e}")
                ws_out[f"{COL['비고']}{r}"] = f"[채점 실패-예외] {type(e).__name__}: {e}"
                continue

            if scoring_type in ("단계형", "선택형"):
                ws_out[f"{COL['충족단계']}{r}"] = result["stage"]
                ws_out[f"{COL['점수']}{r}"] = result["score"]  # 엑셀 수식 대신 직접 계산해서 기입
            elif scoring_type == "감점형":
                v = result["violations"]
                ws_out[f"{COL['위반1']}{r}"] = v["유형1"]
                ws_out[f"{COL['위반2']}{r}"] = v["유형2"]
                ws_out[f"{COL['위반3']}{r}"] = v["유형3"]
                ws_out[f"{COL['점수']}{r}"] = result["score"]  # 엑셀 수식 대신 직접 계산해서 기입
            else:
                ws_out[f"{COL['점수']}{r}"] = result["score"]

            note_parts = ["[자동채점]"]
            if result.get("sufficiency"):
                note_parts.append(f"근거충분성: {result['sufficiency']}")
            if result.get("evidence_pages"):
                note_parts.append(f"근거페이지: {result['evidence_pages']}")
            if result.get("judge_reasoning"):
                note_parts.append(f"근거: {result['judge_reasoning']}")
            if result.get("needs_review"):
                note_parts.append(f"[AI검토필요] {result.get('review_reason', '')}")
            # 엑셀 셀 글자수 한도(32,767자) 방어 + 가독성을 위해 적당히 자름
            ws_out[f"{COL['비고']}{r}"] = " | ".join(note_parts)[:1000]

        output_path = f"{RUBRIC_OUTPUT_ROOT}/{code}_{year}_전체_채점완료.xlsx"
        os.makedirs(RUBRIC_OUTPUT_ROOT, exist_ok=True)
        wb_out.save(output_path)
        print(f"저장 완료: {output_path}")

print("\n전체 채점 완료.")


==================== [SK하이닉스(000660) 2022년] 전체 91개 항목 채점 시작 ====================
  [1/91] P-1-1 ESG 정보공시 방식 채점 중...
  [2/91] P-1-2 ESG 정보공시 주기 채점 중...
  [3/91] P-1-3 ESG 정보공시 범위 채점 중...
  [4/91] P-2-1 ESG 핵심이슈 및 KPI 채점 중...
  [5/91] P-3-1 ESG 정보공시 검증 채점 중...
  [6/91] E-1-1 환경경영 목표 수립 채점 중...
  [7/91] E-1-2 환경경영 추진체계 채점 중...
  [8/91] E-2-1 원부자재 사용량 채점 중...
  [9/91] E-2-2 재생 원부자재 비율 채점 중...
  [10/91] E-3-1 온실가스 배출량 (Scope1 & Scope2) 채점 중...
  [11/91] E-3-2 온실가스 배출량 (Scope3) 채점 중...
  [12/91] E-3-3 온실가스 배출량 검증 채점 중...
  [13/91] E-4-1 에너지 사용량 채점 중...
  [14/91] E-4-2 재생에너지 사용 비율 채점 중...
  [15/91] E-5-1 용수 사용량 채점 중...
  [16/91] E-5-2 재사용 용수 비율 채점 중...
  [17/91] E-6-1 폐기물 배출량 채점 중...
  [18/91] E-6-2 폐기물 재활용 비율 채점 중...
  [19/91] E-7-1 대기오염물질 배출량 채점 중...
  [20/91] E-7-2 수질오염물질 배출량 채점 중...
  [21/91] E-8-1 환경 법/규제 위반 채점 중...
  [22/91] E-9-1 친환경 인증 제품 및 서비스 비율 채점 중...
  [23/91] E-10-1 기후변화 거버넌스 채점 중...
  [24/91] E-10-2 기후변화 중장기 리스크 식별 채점 중...
  [25/91] E-10-3 기후변화 물리 리스크 및 대응 방안 채점 중...
  [26/91]